In [1]:
#import libraries
import os
import datetime as dt
from pathlib import Path
from textwrap import dedent

import arcpy
import numpy as np
import pandas as pd


In [2]:
## COMMANDS TO PREP FOR AUX TRANSIT -- uncomment if not completed

# from create_bus_layers import BusNetwork
# from export_future_highway_network import export_future_highways
# from generate_emme_highway_files import EmmeHighwayNetwork
# from generate_transit_files import EmmeTransitNetwork
# from highway_network import HighwayNetwork



# #export highways into gdbs
# export_future_highways(
#     mhn_gdb_path=mhn_gdb_path,
#     years=[yr for yr in years],
#     output_dir_path=hwy_gdb_out_folder
# )

# #create emme transaction files
# EmmeHighwayNetwork(
#     mhn_all_gdb_path=mhn_all_gdb_path,
#     scenario_years=years,
#     output_folder=hwy_emme_out_folder
# ).generate_hwy_files()

# #export bus into gdbs
# bn = BusNetwork(
#     mhn_gdb_path=mhn_gdb_path,
#     mhn_all_gdb_path=mhn_all_gdb_path,
#     scenario_years=years,
#     output_folder=bus_gdb_out_folder
# )

# bn.create_bn_folder()
# bn.collapse_bus_routes()
# bn.create_bus_layers()

# #create emme transaction files
# EmmeTransitNetwork(
#     bus_network_folder=bus_gdb_out_folder,
#     scenario_years=years,
#     output_folder=bus_emme_out_folder
# ).generate_transit_files()

In [3]:
## I/O global variables
MEM_WORKSPACE = 'memory'
MRN_BATCHIN_FOLDER = r'M:\proj1\nrf\Conformity\c26q2\transit'
YEARS = {2019: 100, 2025: 200, 2030: 300}

PARENT_DIR = r'C:\Users\toleary\0_Tasks\MHN\mfhrn\mfhrn_test'
ZONE_SYSTEM_GDB = os.path.join(PARENT_DIR, "zone_systems.gdb")
MHN_ALL_GDB_PATH = os.path.join(PARENT_DIR, 'output/out_gdb/highway/MHN_all.gdb')
TRANSIT_OUT_FOLDER = os.path.join(PARENT_DIR, 'output/out_gdb/transit')

TRANSACTION_FOLDER = os.path.join(PARENT_DIR, 'output/transaction_files')
if not os.path.isdir(TRANSACTION_FOLDER):
    os.mkdir(TRANSACTION_FOLDER)

In [4]:
# -- FUNCTIONS THAT HELP WITH TESTING -- 
#copy to gdb for manual inspection
def copy_test(in_mem_file, filename : str = 'test'):
    '''given a filepath to a feature class/table, 
    copies the fc/table into source gdb for visual review/inspection.
    name of file is "test" by default, but can optionally change'''
    if arcpy.Exists(f'{MHN_ALL_GDB_PATH}/{filename}'):
        arcpy.management.Delete(f'{MHN_ALL_GDB_PATH}/{filename}')
    arcpy.management.CopyFeatures(in_mem_file, f'{MHN_ALL_GDB_PATH}/{filename}')

#get field names and dtypes of table/fc
def field_dtypes(in_data):
    '''given a string filepath to feature class/table, 
    returns a list of field names and dtypes for inspection'''
    return [[f.name, f.type] for f in arcpy.ListFields(in_data)]

In [5]:
## -- HELPER FUNCTIONS -- ##mem_workspace
#get numpy array of table (non-geometry fields)
def get_table(file_dir, fields=None):
    '''given a string filepath to an arcgis feature class or feature table, returns numpy array 
    of containing data. optional parameter `fields` to specify specific columns to return in array.'''
    if not fields:
        fields = [f.name for f in arcpy.ListFields(file_dir) if f.type != 'Geometry']
    table = arcpy.da.TableToNumPyArray(file_dir, fields, null_value=-9999)
    return table

#delete object if exists
def del_if_exist(file_dir):
    '''deletes arcgis object if exists (nice for files within gdb or "memory" workspace)'''
    if arcpy.Exists(file_dir):
        arcpy.management.Delete(file_dir)

#add zone geometry to points layer
def add_zone_system(in_pts, zone_fc):
    '''given an input points layer and a "zones" feature class, modifies
    input points layer with new field "zone17" populated with the zone
    within which the point resides'''
    print('      - add z17')
    
    arcpy.management.AddSpatialJoin(
        target_features=in_pts,
        # join_features=zone_fc_skinny,
        join_features=zone_fc,
        match_option='WITHIN',
        permanent_join='PERMANENT_FIELDS'
    )

#calculate distance between features (stops and centroids)
def get_dist_stop_to_centroid(stop_fc, centroid_fc):
    '''given a "stops" points layer and "centroid" points layer, calculates
    distances between each and modifies "stops" layer with a new 
    DIST_TO_CENTROID field, filled with distance to centroid in US Survey Feet'''
    print('      - calculating distance to centroid')
    near_table = f'{MEM_WORKSPACE}/near_centroid'
    del_if_exist(near_table)
    arcpy.analysis.GenerateNearTable(
        in_features=stop_fc,
        near_features=centroid_fc,
        out_table=near_table,
        closest='CLOSEST',
        distance_unit='Feet',
        match_fields=[['zone17', 'zone17']]
    )
    arcpy.management.JoinField(
        in_data=stop_fc,
        in_field='OBJECTID',
        join_table=near_table,
        join_field='IN_FID',
        fields=['IN_FID', 'NEAR_DIST']
    )
    arcpy.management.AddField(
        in_table=stop_fc,
        field_name='DIST_TO_CENTROID',
        field_type='FLOAT'
    )
    arcpy.management.CalculateField(
        in_table=stop_fc,
        field='DIST_TO_CENTROID',
        expression='!NEAR_DIST!',
        expression_type='PYTHON3'
    )
    # arcpy.management.DeleteField(
    #     in_table=stop_fc,
    #     drop_field='NEAR_DIST'
    # )

def get_access_link_config(all_stops: str):
    '''access/egress links have a lot of parameters/rules. 
    this function returns a dictionary of the rules, and gets
    used by the get_access_egress_links() function.
    
    mode u: (zone-to-bus) max len 0.55 mi, max of 3 connections per centroid
    mode v: (zone-to-ctarail) max len 0.55 mi, max of 3 connections per centroid in zones where there are cta stops
    mode w: (zone-to-metra) max len 0.55 mi, no restrictions
    mode x: (bus-to-zone) max len 0.55 mi, max of 8 connections in CBD and 2 in non-CBD
    mode y: (ctarail-to-zone) max len 0.55 mi, max of 7 connections in CBD (including zone stop is in) and 3 in non-CBD (including zone stop is in)
    mode z: (metra-to-zone) max len 0.55 mi, no restrictions
    '''
    #set configuration for access/egress links
    # setup access/egress links dictionary
    cta_zones = sorted({
        str(row[0])
        for row in arcpy.da.SearchCursor(all_stops, ['zone17', 'MODE'])
        if row[1] == 'C'
    })
    zones_w_cta_query = (f"zone17 in ({', '.join(cta_zones)})" if cta_zones else None)

    access_links_config = {
        'u': { #access -- home to bus
            'ALLZones': {
                'stops_where': "MODE IN ('P', 'Q', 'L', 'B', 'E')",
                'centroid_where': None,
                'max_connections': 4, #(3 + 1 for zone stop is in, originally 3, then adding zone in case it's not there)
                'max_dist': 0.55 * 5280,
                'aux_link_type': 'access',
                'max_centroid_dist': None
            }
        },
        'v': { #access-- home to cta
            'AllZones': {
                'stops_where': "MODE = 'C'",
                'centroid_where': zones_w_cta_query,
                'max_connections': 3, #(2 + 1 for zone stop is in)
                'max_dist': 0.55 * 5280,
                'aux_link_type': 'access',
                'max_centroid_dist': 0.55 * 5280
            }
        },
        'w': { #access-- home to metra
            'AllZones': {
                'stops_where': "MODE = 'M'",
                'centroid_where': None,
                'max_connections': None,
                'max_dist': 0.55 * 5280,
                'aux_link_type': 'access',
                'max_centroid_dist': 0.55 * 5280
            }
        },
        'x': { #egress-- bus to work
            'CBD': {
                'stops_where': "MODE IN ('P', 'Q', 'L', 'B', 'E')",
                'centroid_where': "zone17 <= 47",
                'max_connections': 9, #(8 + 1 for zone stop is in, originally 8, then adding zone in case it's not there)
                'max_dist': 0.25 * 5280,
                'aux_link_type': 'egress',
                'max_centroid_dist': None
            },
            'NonCBD': {
                'stops_where': "MODE IN ('P', 'Q', 'L', 'B', 'E')",
                'centroid_where': "zone17 > 47",
                'max_connections': 3, #(2 + 1 for zone stop is in, originally 2, then adding zone in case it's not there)
                'max_dist': 0.55 * 5280,
                'aux_link_type': 'egress',
                'max_centroid_dist': None
            }
        },
        'y': { #egress-- cta to work
            'CBD': {
                'stops_where': "MODE = 'C'",
                'centroid_where': "zone17 <= 47",
                'max_connections': 7, #(6 + 1 for zone stop is in)
                'max_dist': 0.55 * 5280,
                'aux_link_type': 'egress',
                'max_centroid_dist': 0.55 * 5280
            },
            'NonCBD': {
                'stops_where': "MODE = 'C'",
                'centroid_where': "zone17 > 47",
                'max_connections': 3, #(2 + 1 for zone stop is in)
                'max_dist': 0.55 * 5280,
                'aux_link_type': 'egress',
                'max_centroid_dist': 0.55 * 5280
            }
        },
        'z': { #egress-- metra to work
            'AllZones': {
                'stops_where': "MODE = 'M'",
                'centroid_where': None,
                'max_connections': None,
                'max_dist': 0.55 * 5280,
                'aux_link_type': 'egress',
                'max_centroid_dist': 0.55 * 5280
            }
        }
    }
    
    return access_links_config


In [6]:
#prepare bus and rail stops -- grab stop info from itineraries and join to zone information
def prepare_rail_stops(scen_code, zone17_fc, centroid_fc, mrn_batchin_folder=MRN_BATCHIN_FOLDER):
    '''given a MRN transaction file location and a scenario code, will read info from 
    MRN batchin and return a dictionary of information for that scenario of the format:
    {
        time_of_day_code: {
            'itin': <itinerary_transaction_file_location>,
            'net': <network_transaction_file_location>,
            'rail_stops_point': pd.DataFrame of rail stops, both mode and location
        }
    }
    '''
    print(f'prepping rail stops: scenario {scen_code}')
    scen_1dig = str(scen_code)[0]
    try:
        mrn_year_dir = next(d for d in Path(mrn_batchin_folder).iterdir() if d.stem == str(scen_code))
    except FileNotFoundError as e:
        print(f'Couldnt find MRN dir. Check filepath. Are you connected to the VPN? \n {e}')
        raise
    rail_itins = sorted([fl for fl in mrn_year_dir.iterdir() if 'rail.itinerary_' in fl.name])
    rail_nets = sorted([fl for fl in mrn_year_dir.iterdir() if 'rail.network_' in fl.name])
    rail_tods = {int(str(i)[-1]): {'itin': i, 'net': n} for i, n in zip(rail_itins, rail_nets)}
    print(f'  - rail tods: {[k for k in rail_tods]}')
    
    for tod, tod_files in rail_tods.items():
        # print(f'    - processing {tod}')
        rail_itin = tod_files['itin']
        rail_net = tod_files['net']
        
        #GET RAIL STOP INFO
        print(f'      - itin: {rail_itin}')
        with open(rail_itin, 'r') as itin_file:
            itin_lines = itin_file.readlines()
        #grab stops from itin transaction file
        stop_nodes = []
        next_node_stops = True
        current_line = None
        current_mode = None
        for line in itin_lines:
            #parse mode, node, and dwell code from line
            line_info = [i.strip() for i in line.strip().split() if i]
            #if a new line is being initialized, mark "current_line" and "current_mode"
            if line.lstrip().startswith('a'):
                current_line = line_info[1].replace("'", "")
                current_mode = line_info[2]
                next_node_stops = True
            #parse node and dwell code from line
            elif line_info[0].isnumeric():
                #log stop if true
                if next_node_stops:
                    stop_nodes.append([current_line, current_mode, line_info[0]])
                #check dwt for next node
                if len(line_info) > 1 and 'lay' not in line_info[-1]:
                    dwt_next_node = line_info[1]
                    if '#0' in dwt_next_node:
                        next_node_stops = False
                    else:
                        next_node_stops = True

        stop_nodes = pd.DataFrame(data=stop_nodes, columns=['line', 'mode', 'node'])
        stop_nodes['stop'] = 1
        stop_nodes = stop_nodes[['mode','node', 'stop']].drop_duplicates()
        
        #grab coordinates from network transaction file
        # print(f'      - net: {rail_net}')
        with open(rail_net, 'r') as net_file:
            net_lines = net_file.readlines()

        #find nodes in network file
        node_header = next(
            index for index, line in enumerate(net_lines) 
            if line.lstrip().startswith('t') and 'node' in line
        )
        node_start = node_header + 1
        node_rows = []
        for line in net_lines[node_start:]:
            if line.lstrip().startswith('t'):
                break
            elif line.lstrip().startswith('a'):
                line_info = [l for l in line.strip().split(' ') if l]
                node_rows.append(line_info)
        node_cols = ['a', 'node', 'x', 'y', 'ui1', 'ui2', 'ui3', 'label']
        node_df = pd.DataFrame(node_rows, columns=node_cols)
        node_df = node_df[['node', 'x', 'y']]
        node_df = node_df.merge(stop_nodes, on='node', how='inner')
        node_df.rename(
            columns={c: c.upper() for c in node_df.columns},
            inplace=True
        )
        
        #bring rail stops into point geometry, join with zones
        rail_stops_table = f'{MEM_WORKSPACE}/rail_stops'
        rail_stops_point = f'{MEM_WORKSPACE}/rail_stops_pt_{tod}'
        rail_stops_dtypes = {'NODE':np.int32, 'X':np.float64, 'Y':np.float64, 'MODE':"U1"}
        del_if_exist(rail_stops_table)
            
        #make rail stops into table
        rail_stop_array_for_arc = (
            node_df[['X','Y','MODE','NODE']]
            .to_records(
                index=False,
                column_dtypes=rail_stops_dtypes
            )
        )
        arcpy.da.NumPyArrayToTable(
            rail_stop_array_for_arc,
            rail_stops_table
        )
        arcpy.management.DeleteIdentical(
            rail_stops_table,
            fields=['NODE', 'MODE']
        )
        #make table into points
        del_if_exist(rail_stops_point)
        arcpy.management.XYTableToPoint(
            in_table=rail_stops_table,
            out_feature_class=rail_stops_point,
            x_field='X',
            y_field='Y',
            coordinate_system=arcpy.SpatialReference(26771)
        )
        del_if_exist(rail_stops_table)
        
        add_zone_system(rail_stops_point, zone17_fc)
        get_dist_stop_to_centroid(rail_stops_point, centroid_fc)
        
        rail_tods[tod]['rail_stops_point'] = rail_stops_point
        
    return rail_tods
def prepare_bus_stops(scen_code, zone17_fc, centroid_fc, transit_out_folder=TRANSIT_OUT_FOLDER):
    '''creates dictionary of information for a given scenario for each bus time-of-day period, of the format:
    {
        time_of_day_code: {
            'itin': <itinerary_transaction_file_location>,
            'net': <network_transaction_file_location>,
            'bus_stops_point': pd.DataFrame of rail stops, both mode and location
        }
    }
    '''
    
    print(f'prepping bus stops: scenario {scen_code}')
    #bring bus stops into point geometry
    scen_1dig = str(scen_code)[0]
    
    #find feature classes (itin_1 through itin_4, stored in feature datasets TOD_1–TOD_4)  
    cindy_bus_gdb = f'{transit_out_folder}/SCENARIO_{scen_1dig}.gdb'
    arcpy.env.workspace = cindy_bus_gdb
    list_fcs = []
    tod_dss = [f for f in arcpy.ListDatasets()]
    for ds in tod_dss:
        arcpy.env.workspace = os.path.join(cindy_bus_gdb, ds)
        try:
            list_fcs += [f for f in arcpy.ListFeatureClasses()]
        except TypeError:
            print(f'Warning: No itin dataset found in {ds}')
            continue
    bus_itin_disk = [f for f in list_fcs if 'scen_itin' in f]
    bus_tods = {int(nam[-1]): {'itin': f'{cindy_bus_gdb}/{nam}'} for nam in bus_itin_disk}
    # print(f'  - bus tods: {[b for b in bus_tods]}')
    arcpy.env.workspace = 'memory'

    for tod, bus_info in bus_tods.items():
        print(f'    - tod {tod}')
        bus_itin = bus_info['itin']
        
        #only copy over itin segments with stop on itin_b
        mem_bus_itin = f"{MEM_WORKSPACE}/bus_itin_{tod}"
        del_if_exist(mem_bus_itin)
        arcpy.analysis.Select(
            in_features=bus_itin,
            out_feature_class=mem_bus_itin,
            where_clause="DWELL_CODE = '0'"
        )
        arcpy.management.AddField(
            in_table=mem_bus_itin,
            field_name='MODE',
            field_type='TEXT'
        )
        arcpy.management.CalculateField(
            in_table=mem_bus_itin,
            field='MODE',
            expression='str(!TRANSIT_LINE!)[0].upper()',
            expression_type='PYTHON3'
        )
        
        arcpy.management.DeleteIdentical(
            in_dataset=mem_bus_itin,
            fields=['ITIN_B', 'MODE', 'TRANSIT_LINE']
        )
        
        #get df of itin-b and mode values to search for later
        itin_mode_np = get_table(mem_bus_itin)[['ITIN_B', 'TRANSIT_LINE', 'MODE']]
        itin_mode = pd.DataFrame(itin_mode_np)
        # itin_mode['MODE'] = itin_mode['TRANSIT_LINE'].str.strip().str.upper().str[0]
        # itin_mode = itin_mode[['ITIN_B', 'TRANSIT_LINE', 'MODE']].copy()
        
        #extract stops as point geometry
        bus_stops_point = f'{MEM_WORKSPACE}/bus_stop_point_{tod}'
        del_if_exist(bus_stops_point)
        arcpy.management.CreateFeatureclass(
            MEM_WORKSPACE, 
            os.path.basename(bus_stops_point),
            geometry_type='POINT',
            spatial_reference=arcpy.SpatialReference(26771)
        )
        arcpy.management.AddField(
            in_table=bus_stops_point,
            field_name='NODE',
            field_type='LONG'
        )
        arcpy.management.AddField(
            in_table=bus_stops_point,
            field_name='MODE',
            field_type='TEXT'
        )
        arcpy.management.AddField(
            in_table=bus_stops_point,
            field_name='TRANSIT_LINE',
            field_type='TEXT'
        )
        
        all_nodes = os.path.join(MHN_ALL_GDB_PATH, 'hwynode_all')
        with arcpy.da.SearchCursor(all_nodes, ['NODE', 'SHAPE@']) as sc:
            with arcpy.da.InsertCursor(bus_stops_point, ['NODE','MODE', 'TRANSIT_LINE', 'SHAPE@X', 'SHAPE@Y']) as ic:
                for node_row in sc: 
                    node = node_row[0]
                    node_shape = node_row[1].firstPoint
                    stop_x = node_shape.X
                    stop_y = node_shape.Y
                    is_stop = itin_mode.loc[itin_mode['ITIN_B'].astype(int)==int(node)]
                    if len(is_stop) > 0:
                        for _, stop_row in is_stop.iterrows():
                            mode = stop_row['MODE']
                            tr_line = stop_row['TRANSIT_LINE']
                            ic.insertRow([node, mode, tr_line, stop_x, stop_y])
        
        add_zone_system(bus_stops_point, zone17_fc)
        get_dist_stop_to_centroid(bus_stops_point, centroid_fc)
        
        bus_tods[tod]['bus_stops_point'] = bus_stops_point
        
    return bus_tods

In [18]:
def generate_auxiliary_transit():
    '''main function to generate all auxiliary link information 
    for given scenario years
    
    (1) get zone and centroid information
    (2) for each scenario year submitted:
        (a) get all time-of-day stop info for bus and rail
        (b) for each time-of-day period:
            (i) combine stop info into one table (all_stops)
            (ii) `get_transfer_links()`: calculate transfer links (mode-to-mode)
            (iii) `get_access_egress_links()`: calculate access/egress links (zone-to-mode)
            (iv) `get_centroid_connections()`: get all zone-centroid pairs (all transit stops connect to zone they reside in)
            (v) `write_transaction():` write transaction files
    
    '''
    
    #get copy of zone17 polygons
    zone17_fc = f'{MEM_WORKSPACE}/z17'
    del_if_exist(zone17_fc)
    arcpy.management.CopyFeatures(
        in_features=os.path.join(ZONE_SYSTEM_GDB, 'zones17'),
        out_feature_class=zone17_fc
    )

    #get copy of centroid points
    centroid_fc = f'{MEM_WORKSPACE}/centroid'
    del_if_exist(centroid_fc)
    arcpy.analysis.Select(
        in_features=os.path.join(MHN_ALL_GDB_PATH, 'hwynode_all'),
        out_feature_class=centroid_fc,
        where_clause='NODE = zone17'
    )
    
    for year, scen_code in YEARS.items():
        
        #get all the transit information for the given scenario code
        scen_bus_tods = prepare_bus_stops(scen_code, zone17_fc, centroid_fc)
        scen_rail_tods = prepare_rail_stops(scen_code, zone17_fc, centroid_fc)
        
        #find times of day -- ensure bus and rail tods match up
        bus_tods = [c for c in scen_bus_tods]
        rail_tods = [c for c in scen_rail_tods]
        all_tods = set(bus_tods + rail_tods)
        if set(bus_tods) != all_tods or set(rail_tods) != all_tods:
            raise ValueError(f'bus and rail tods do not match. bus: {bus_tods}; rail: {rail_tods}.')
        
        #generate aux links for each tod
        for tod in all_tods:
            print(f'creating aux links for tod {tod}')
            
            #PREP -- make bus, rail, and all_stops fc's
            bus = f'{MEM_WORKSPACE}/bus'
            del_if_exist(bus)
            
            arcpy.management.CopyFeatures(
                in_features=scen_bus_tods[tod]['bus_stops_point'],
                out_feature_class=bus
            )
            
            arcpy.management.DeleteField(
                in_table=bus,
                drop_field=['Shape', 'NODE', 'MODE', 'TRANSIT_LINE', 'DIST_TO_CENTROID', 'zone17'],
                method='KEEP_FIELDS' #keep only fields above
            )

            rail = f'{MEM_WORKSPACE}/rail_{scen_code}_{tod}'
            del_if_exist(rail)

            arcpy.management.CopyFeatures(
                in_features=scen_rail_tods[tod]['rail_stops_point'],
                out_feature_class=rail,
            )
            arcpy.management.DeleteField(
                in_table=rail,
                drop_field=['Shape', 'NODE', 'MODE', 'DIST_TO_CENTROID', 'zone17'],
                method='KEEP_FIELDS' #keep only fields above
            )


            all_stops = f'{MEM_WORKSPACE}/all_stops'
            del_if_exist(all_stops)
            arcpy.management.CreateFeatureclass(
                out_path=MEM_WORKSPACE,
                out_name=os.path.basename(all_stops),
                template=bus,
                spatial_reference=arcpy.SpatialReference(26771)
            )
            arcpy.management.Append(
                inputs=[bus, rail],
                target=all_stops,
                schema_type='NO_TEST'
            ) 
            
            transfer_links = get_transfer_links(scen_code, year, all_stops)
            acc_egr_links = get_access_egress_links(all_stops, centroid_fc)
            stop_centroid = get_centroid_connections(all_stops)
            
            all_aux = pd.concat([transfer_links, acc_egr_links, stop_centroid])
            
            #enforce max_connections from access_link_config
            max_connects = []
            for mod, geog_config in get_access_link_config(all_stops).items():
                for geog, config in geog_config.items():
                    max_connections = config['max_connections']
                    max_connects.append([mod, max_connections])
            
            
            all_aux.to_csv(rf'C:\Users\toleary\OneDrive - Chicago Metropolitan Agency for Planning\Desktop\test_{scen_code}_{tod}.csv')
       
def get_centroid_connections(all_stops: str) -> pd.DataFrame:
    '''function returns a pandas dataframe of all stop-centroid pairs (where stops and centroids are in same zone)'''
    stops = pd.DataFrame(get_table(all_stops))
    # stops.rename(columns={c: c.lower() for c in stops.columns}, inplace=True)
    stop_centroid_pairs = stops.drop_duplicates(['zone17', 'NODE', 'MODE'])
    stop_centroid_pairs.rename(columns={'DIST_TO_CENTROID': 'length'}, inplace=True)
    
    #for df.apply(), make map for auxiliary mode
    def aux_mode_home_to_stop(x):
        if x.upper() in ['B', 'E', 'P', 'Q', 'L']:
            return 'u' #home to bus
        elif x.upper() == 'C':
            return 'v' #home to cta rail
        elif x.upper() == 'M':
            return 'w' #home to metra rail
        else:
            return ''
    def aux_mode_stop_to_home(x):
        if x.upper() in ['B', 'E', 'P', 'Q', 'L']:
            return 'x'
        elif x.upper() == 'C':
            return 'y'
        elif x.upper() == 'M':
            return 'z'
        else:
            return ''
    
    #for home-to-stop direction
    stop_centroid_pairs_ab = stop_centroid_pairs.copy()
    stop_centroid_pairs_ab[['ANODE', 'BNODE']] = stop_centroid_pairs_ab[['zone17', 'NODE']]
    stop_centroid_pairs_ab['aux_mode'] = stop_centroid_pairs_ab['MODE'].apply(aux_mode_home_to_stop)
    
    #for stop-to-home direction
    stop_centroid_pairs_ba = stop_centroid_pairs.copy()
    stop_centroid_pairs_ba[['ANODE', 'BNODE']] = stop_centroid_pairs[['NODE', 'zone17']]
    stop_centroid_pairs_ba['aux_mode'] = stop_centroid_pairs_ba['MODE'].apply(aux_mode_stop_to_home)
    
    stop_centroid_pairs = pd.concat([stop_centroid_pairs_ab, stop_centroid_pairs_ba])
    stop_centroid_pairs.drop_duplicates(['ANODE','BNODE','aux_mode'], inplace=True)

    return stop_centroid_pairs[['ANODE','BNODE','aux_mode', 'length']]
    
    stop_centroid_pairs = pd.concat([stop_centroid_pairs_ab, stop_centroid_pairs_ba])[['ANODE', 'BNODE']]
    return stop_centroid_pairs

#find mode-to-mode transfers
def get_transfer_links(
    scen_code: int | str,
    year: int | str,
    all_stops: str,
    zone17_fc: str = os.path.join(ZONE_SYSTEM_GDB, 'zones17'),
    # trnt_gdb_out_folder: str = TRANSIT_OUT_FOLDER,
) -> pd.DataFrame:
    '''function to derive transfer links for modes b, c, and m:
        - b -- bus to bus
        - c -- bus to cta rail
        - m -- bus to metra
    Returns a pandas DataFrame object of transfer links
    Parameters:
    all_stops -- feature class of all stops (bus, rail, and metra)
    zone17_fc -- feature class of all modeling zones
    mhn_fc -- feature class of highway links (HWYLINK_{year})
    '''
    # scen_1dig = str(scen_code)[0]
    
    #these functions defined below
    transfer_links = pd.concat([
        get_mode_b(year, zone17_fc),
        get_mode_c(all_stops),
        get_mode_m(all_stops)
    ])
    
    return transfer_links

#def get mode b, c, m

#def get_mode_b() -- bus-to-bus transfer links
def get_mode_b(year, zone17_fc):    
    '''function taking mhn_all_gdb and outputting the VDF1 links that are zones 1-78-- these become mode b'''
    # mode b
    #   - all VDF 1 in zone 1-78
    mhn_fc = f'{MHN_ALL_GDB_PATH}/HWYLINK_{year}'
    mhn_vdf1 = f'{MEM_WORKSPACE}/mhn_vdf1'
    del_if_exist(mhn_vdf1)
    arcpy.analysis.Select(
        in_features=mhn_fc,
        out_feature_class=mhn_vdf1,
        where_clause="TYPE1 = '1' OR TYPE2 = '1'"
    )
    central_zones = f'{MEM_WORKSPACE}/central_zones'
    del_if_exist(central_zones)
    arcpy.analysis.Select(
        in_features=zone17_fc,
        out_feature_class=central_zones,
        where_clause='zone17 <= 78'
    )
    arcpy.management.AddSpatialJoin(
        target_features=mhn_vdf1,
        join_features=central_zones,
        match_option='WITHIN',
        permanent_join='PERMANENT_FIELDS'
    )
    
    arcpy.management.AddField(
        in_table=mhn_vdf1,
        field_name='length',
        field_type='FLOAT'
    )
    
    arcpy.management.CalculateGeometryAttributes(
        in_features=mhn_vdf1,
        geometry_property=[['length', 'LENGTH']],
        length_unit='FEET_US'
    )
    
    
    mhn_in_central = pd.DataFrame(get_table(mhn_vdf1))
    #filter out non-joins (-9999, as defined in get_table() function)
    b_links = mhn_in_central[mhn_in_central['zone17']!= -9999].copy()
    b_links_ab = b_links[['ANODE', 'BNODE', 'length', 'zone17']].copy()
    b_links_ba = b_links[['ANODE', 'BNODE', 'length', 'zone17']].copy()
    b_links_ba[['BNODE', 'ANODE']] = b_links_ba[['ANODE','BNODE']]
    b_links = pd.concat([b_links_ab, b_links_ba])
    b_links['aux_mode'] = 'b'
    b_links = b_links[['aux_mode', 'ANODE', 'BNODE', 'length', 'zone17']]
    # b_links['link'] = b_links['ANODE'].astype(str) + '-' + b_links['BNODE'].astype(str)
    
    return b_links

#def get_mode_c() -- bus-to-L transfer links
def get_mode_c(all_stops) -> pd.DataFrame:
    '''function taking all transit stops in MHN/MRN as parameter and outputting
    auxiliary transit mode 'c' links in a pandas DataFrame.
    mode 'c' links: bus to cta rail
        - 1/8 mile inside CBD 
        - 1/2 mile outside CBD (but only one connection per bus route allowed)
    '''
    # -- 1/8 in CBD
    cbd_bus = f'{MEM_WORKSPACE}/cbd_bus_w_lines'
    notcbd_bus = f'{MEM_WORKSPACE}/bus_w_lines'
    c_cbd_neartable = f'{MEM_WORKSPACE}/c_cbd_neartable'
    c_noncbd_neartable = f'{MEM_WORKSPACE}/c_noncbd_neartable'
    cta = f'{MEM_WORKSPACE}/cta'
    working_data = [cbd_bus, notcbd_bus, c_cbd_neartable, c_noncbd_neartable, cta]
    for data in working_data:
        del_if_exist(data)

    #get cta stops
    arcpy.analysis.Select(
        in_features=all_stops,
        out_feature_class=cta,
        where_clause="MODE = 'C'"
    )

    # mode c -- cbd (1/8 mile)
    arcpy.analysis.Select(
        in_features=all_stops,
        out_feature_class=cbd_bus,
        where_clause="zone17 <= 47 AND MODE IN ('P', 'Q', 'L', 'B', 'E')"
    )

    arcpy.analysis.GenerateNearTable(
        in_features=cbd_bus,
        near_features=cta,
        out_table=c_cbd_neartable,
        search_radius= (1/8) * 5280,
        closest='ALL',
        distance_unit='Feet'
    )

    #get node info from cbd bus fc
    arcpy.management.JoinField(
        in_data=c_cbd_neartable,
        in_field='IN_FID',
        join_table=cbd_bus,
        join_field='OBJECTID',
        fields=['NODE', 'zone17']
    )

    #alterfield throws errors, so add + calculate + delete
    arcpy.management.AddField(
        in_table=c_cbd_neartable,
        field_name='bus_node',
        field_type='LONG'
    )
    arcpy.management.CalculateField(
        in_table=c_cbd_neartable,
        field='bus_node',
        expression='!NODE!',
        expression_type='PYTHON3'
    )
    arcpy.management.DeleteField(
        in_table=c_cbd_neartable,
        drop_field='NODE'
    )

    #get node info from the other bus nodes
    arcpy.management.JoinField(
        in_data=c_cbd_neartable,
        in_field='NEAR_FID',
        join_table=cta,
        join_field='OBJECTID',
        fields=['NODE', 'zone17']
    )
    arcpy.management.AddField(
        in_table=c_cbd_neartable,
        field_name='cta_node',
        field_type='LONG'
    )
    arcpy.management.CalculateField(
        in_table=c_cbd_neartable,
        field='cta_node',
        expression='!NODE!',
        expression_type='PYTHON3'
    )
    arcpy.management.DeleteField(
        in_table=c_cbd_neartable,
        drop_field='NODE'
    )
    
    #get data, make links in both directions
    c_cbd = pd.DataFrame(get_table(c_cbd_neartable))
    c_cbd.rename(columns={'NEAR_DIST':'length'}, inplace=True)
    c_cbd['aux_mode'] = 'c'
    c_cbd[['bus_node', 'cta_node']].drop_duplicates(inplace=True)
    #make bus to rail direction
    c_cbd_ab = c_cbd.copy()
    c_cbd_ab[['ANODE', 'BNODE']] = c_cbd_ab[['bus_node', 'cta_node']]
    #make rail to bus direction
    c_cbd_ba = c_cbd.copy()
    c_cbd_ba[['ANODE', 'BNODE']] = c_cbd_ba[['cta_node', 'bus_node']]
    #put together
    c_cbd = pd.concat([c_cbd_ab, c_cbd_ba]).drop_duplicates(['ANODE','BNODE'])[['aux_mode', 'ANODE','BNODE', 'length', 'zone17']]

    # --
    # mode c not in cbd -- 1/2 mi, but only the shortest link for each route is retained 
    arcpy.analysis.Select(
        in_features=all_stops,
        out_feature_class=notcbd_bus,
        where_clause="zone17 > 47 AND MODE IN ('P', 'Q', 'L', 'B', 'E')"
    )

    arcpy.analysis.GenerateNearTable(
        in_features=notcbd_bus,
        near_features=cta,
        out_table=c_noncbd_neartable,
        search_radius= (1/2) * 5280,
        closest='ALL',
        distance_unit='Feet'
    )

    #get node info from cbd bus fc
    arcpy.management.JoinField(
        in_data=c_noncbd_neartable,
        in_field='IN_FID',
        join_table=notcbd_bus,
        join_field='OBJECTID',
        fields=['NODE', 'TRANSIT_LINE']
    )

    #`alterfield` throws errors, so `add` + `calculate` + `delete`
    arcpy.management.AddField(
        in_table=c_noncbd_neartable,
        field_name='bus_node',
        field_type='LONG'
    )
    arcpy.management.CalculateField(
        in_table=c_noncbd_neartable,
        field='bus_node',
        expression='!NODE!',
        expression_type='PYTHON3'
    )
    arcpy.management.DeleteField(
        in_table=c_noncbd_neartable,
        drop_field='NODE'
    )

    #get node info from the other bus nodes
    arcpy.management.JoinField(
        in_data=c_noncbd_neartable,
        in_field='NEAR_FID',
        join_table=cta,
        join_field='OBJECTID',
        fields=['NODE', 'zone17']
    )
    arcpy.management.AddField(
        in_table=c_noncbd_neartable,
        field_name='cta_node',
        field_type='LONG'
    )
    arcpy.management.CalculateField(
        in_table=c_noncbd_neartable,
        field='cta_node',
        expression='!NODE!',
        expression_type='PYTHON3'
    )
    arcpy.management.DeleteField(
        in_table=c_noncbd_neartable,
        drop_field='NODE'
    )

    #get data, only keep best connection per transit line
    c_noncbd = pd.DataFrame(get_table(c_noncbd_neartable))
    c_noncbd['aux_mode'] = 'c'
    c_noncbd = (
        c_noncbd
        .sort_values(['cta_node','TRANSIT_LINE','NEAR_DIST'])
        .drop_duplicates(['cta_node','TRANSIT_LINE'], keep='first')
    )
    c_noncbd.rename(columns={'NEAR_DIST':'length'}, inplace=True)
    #make bus to rail direction
    c_noncbd_ab = c_noncbd.copy()
    c_noncbd_ab[['ANODE', 'BNODE']] = c_noncbd[['bus_node', 'cta_node']]
    #make rail to bus direction
    c_noncbd_ba = c_noncbd.copy()
    c_noncbd_ba[['ANODE', 'BNODE']] = c_noncbd[['cta_node', 'bus_node']]
    c_noncbd = pd.concat([c_noncbd_ab, c_noncbd_ba])[['aux_mode', 'ANODE', 'BNODE', 'length', 'zone17']]

    for data in working_data:
        del_if_exist(data)
        
    mode_c = pd.concat([c_cbd, c_noncbd])
    
    return mode_c

#def get_mode_m() -- bus-to-Metra transfer links
def get_mode_m(all_stops: str):
    '''function taking all transit stops and outputting pandas dataframe of bus-to-metra links (mode m)
    mode m: bus-to-metra-- 
        - 1/4 mile for CTA bus to Metra
        - 0.55 mile for Pace bus to Metra
    '''
    #name all arc data that will be used
    cta_bus = f'{MEM_WORKSPACE}/cta_bus'
    pace_bus = f'{MEM_WORKSPACE}/pace_bus'
    metra = f'{MEM_WORKSPACE}/metra'
    cta_metra_near = f'{MEM_WORKSPACE}/cta_metra_near'
    pace_metra_near = f'{MEM_WORKSPACE}/pace_metra_near'
    working_data = [cta_bus, pace_bus, metra, cta_metra_near, pace_metra_near]
    for data in working_data:
        del_if_exist(data)

    arcpy.analysis.Select(
        in_features=all_stops,
        out_feature_class=cta_bus,
        where_clause="MODE IN ('C', 'E')"
    )

    arcpy.analysis.Select(
        in_features=all_stops,
        out_feature_class=pace_bus,
        where_clause="MODE IN ('P', 'Q', 'L')"
    )

    arcpy.analysis.Select(
        in_features=all_stops,
        out_feature_class=metra,
        where_clause="MODE = 'M'"
    )

    #CTA bus to metra
    arcpy.analysis.GenerateNearTable(
        in_features=cta_bus,
        near_features=metra,
        out_table=cta_metra_near,
        search_radius= (1/4) * 5280,
        closest='ALL',
        distance_unit='Feet'
    )

    #Pace bus to metra
    arcpy.analysis.GenerateNearTable(
        in_features=pace_bus,
        near_features=metra,
        out_table=pace_metra_near,
        search_radius= 0.55 * 5280,
        distance_unit='Feet'
    )

    mode_m = []
    for neartable, trnt in [[cta_metra_near, cta_bus], [pace_metra_near, pace_bus]]:
        #add bus_node and metra_node fields
        arcpy.management.AddField(
            in_table=neartable,
            field_name='bus_node',
            field_type='LONG'
        )

        arcpy.management.AddField(
            in_table=neartable,
            field_name='metra_node',
            field_type='LONG'
        )

        #join original cta/pace node info to neartable
        arcpy.management.JoinField(
            in_data=neartable,
            in_field='IN_FID',
            join_table=trnt,
            join_field='OBJECTID',
            fields=['NODE', 'zone17']
        )

        arcpy.management.CalculateField(
            in_table=neartable,
            field='bus_node',
            expression='!NODE!',
            expression_type='PYTHON3'
        )

        arcpy.management.DeleteField(
            in_table=neartable,
            drop_field='NODE'
        )
        
        #join original metra node info to neartable
        arcpy.management.JoinField(
            in_data=neartable,
            in_field='NEAR_FID',
            join_table=metra,
            join_field='OBJECTID',
            fields=['NODE', 'zone17']
        )

        arcpy.management.CalculateField(
            in_table=neartable,
            field='metra_node',
            expression='!NODE!',
            expression_type='PYTHON3'
        )

        arcpy.management.DeleteField(
            in_table=neartable,
            drop_field='NODE'
        )

        mode_m.append(pd.DataFrame(get_table(neartable)))

    m_links = pd.concat(mode_m)

    for data in working_data:
        del_if_exist(data)

    m_links['aux_mode'] = 'm'
    m_links.rename(columns={'NEAR_DIST':'length'}, inplace=True)
    m_links_ab = m_links.copy()
    m_links_ab[['ANODE', 'BNODE']] = m_links_ab[['bus_node', 'metra_node']]
    m_links_ba = m_links.copy()
    m_links_ba[['ANODE','BNODE']] = m_links_ba[['metra_node', 'bus_node']]
    m_links = pd.concat([m_links_ab, m_links_ba])[['aux_mode', 'ANODE', 'BNODE', 'length', 'zone17']]

    return m_links

#get access/egress links -- u, v, w, x, y, and z
def get_access_egress_links(
    all_stops: str,
    centroid_fc: str,
    ) -> pd.DataFrame:
    '''wrapper function for find_access_links(), which returns access/egress links for 
    a given access/egress mode (u, v, w, x, y, or z) according to parameters set in 'access_links_config'
    '''
    access_links_config = get_access_link_config(all_stops)
    access_links = []
    for acc_mode, acc_mode_config in access_links_config.items():
        for query in acc_mode_config.values():
            # print(f'{aux_mode} {geog}...')
            # print(f'  - stops_where = {query['stops_where']}')
            # print(f'  - centroid_where = {query['centroid_where']}')
            # print(f'  - max_connections = {query['max_connections']}')
            # print(f'  - max_dist = {query['max_dist']}')
            link_df = find_access_links(
                all_stops=all_stops,
                centroid_fc=centroid_fc,
                aux_mode_value=acc_mode,
                stops_where=query['stops_where'],
                centroid_where=query['centroid_where'],
                max_connections=query['max_connections'],
                max_dist=query['max_dist']
            )
            if query['aux_link_type'] == 'access':
                link_df[['ANODE', 'BNODE']] = link_df[['zone17', 'node']]
                # link_df['link'] = link_df['node'].astype(str) + '-' + link_df['zone17'].astype(str)
            else:
                link_df[['ANODE','BNODE']] = link_df[['node', 'zone17']]
                # link_df['link'] = link_df['zone17'].astype(str) + '-' + link_df['node'].astype(str)
            access_links.append(link_df)
    access_links = pd.concat(access_links)[['aux_mode', 'ANODE', 'BNODE', 'length', 'zone17']]
    
    return access_links

#helper to find access/egress links given some specific parameter info
def find_access_links(
    all_stops: str,
    centroid_fc: str,
    aux_mode_value: str | None = None,
    stops_where: str | None = None,
    centroid_where: str | None = None,
    max_connections: int | None = None,
    max_dist: float | None = None
) -> pd.DataFrame:
    '''function to find auxiliary access/egress transit connections (between centroids and transit stops). 
    Returns a pandas dataframe of the auxiliary transit connections. 
    this tool is meant to be run with specific parameters/rules. passed within wrapper function 'get_access_egress_links' 
    along with a config dictionary that specifies each of the following parameters:
    Parameters:
    all_stops: str representing path to an arcpy feature class containing all stop locations for all transit modes
    centroid_fc: str representing path to an arcpy feature class containing all centroids
    mode_value: str representing the auxiliary transit mode value
    stops_where: str of SQL statement (after "WHERE") defining subset of all_stops to analyze
    centroid_where: str of SQL statement (after "WHERE") defining subset of centroid_fc to analyze
    max_connections: integer of maximum connections available to centroids (e.g., "no more than 6 transit stops per CBD centroid")
    max_dist: maximum distance (in US Survey Feet) that a stop can be from the centroid
    '''

    selected_centroids = f'{MEM_WORKSPACE}/sel_centroids'
    selected_stops = f'{MEM_WORKSPACE}/sel_stops'
    neartable_stops = f'{MEM_WORKSPACE}/neartable_stops'
    working_data = [selected_centroids, selected_stops, neartable_stops]
    
    #clean memory before and after execution
    for data in working_data:
        del_if_exist(data)
        
    arcpy.analysis.Select(
        in_features=centroid_fc, 
        out_feature_class=selected_centroids,
        where_clause=centroid_where
    )

    arcpy.analysis.Select(
        in_features=all_stops,
        out_feature_class=selected_stops,
        where_clause=stops_where
    )
    
    arcpy.analysis.GenerateNearTable(
        in_features=selected_centroids,
        near_features=selected_stops,
        out_table=neartable_stops,
        search_radius=max_dist,
        closest='ALL',
        closest_count=max_connections,
        distance_unit='Feet'
    )
    
    arcpy.management.JoinField(
        in_data=neartable_stops,
        in_field='IN_FID',
        join_table=selected_centroids,
        join_field='OBJECTID',
        fields=['zone17']
    )

    arcpy.management.JoinField(
        in_data=neartable_stops,
        in_field='NEAR_FID',
        join_table=selected_stops,
        join_field='OBJECTID',
        fields=['NODE', 'MODE']
    )
    
    out_table = pd.DataFrame(arcpy.da.TableToNumPyArray(neartable_stops, '*'))
    
    out_table.rename(columns={'NEAR_DIST':'length'}, inplace=True)
    out_table.rename(columns={c:c.lower() for c in out_table.columns}, inplace=True)
    out_table = out_table[['mode', 'zone17', 'node', 'length']].copy()
    out_table['aux_mode'] = aux_mode_value
    
    #clean memory before and after execution
    for data in working_data:
        del_if_exist(data)
    
    return out_table

In [19]:
generate_auxiliary_transit()

prepping bus stops: scenario 100
    - tod 1
      - add z17
      - calculating distance to centroid
    - tod 2
      - add z17
      - calculating distance to centroid
    - tod 3
      - add z17
      - calculating distance to centroid
    - tod 4
      - add z17
      - calculating distance to centroid
prepping rail stops: scenario 100
  - rail tods: [1, 2, 3, 4]
      - itin: M:\proj1\nrf\Conformity\c26q2\transit\100\rail.itinerary_1
      - add z17
      - calculating distance to centroid
      - itin: M:\proj1\nrf\Conformity\c26q2\transit\100\rail.itinerary_2
      - add z17
      - calculating distance to centroid
      - itin: M:\proj1\nrf\Conformity\c26q2\transit\100\rail.itinerary_3
      - add z17
      - calculating distance to centroid
      - itin: M:\proj1\nrf\Conformity\c26q2\transit\100\rail.itinerary_4
      - add z17
      - calculating distance to centroid
creating aux links for tod 1


c:\Users\toleary\AppData\Local\Programs\ArcGIS\Pro\Resources\ArcPy\arcpy\__init__.py:2969: Warning: The content in the specified field(s) exceeds the defined length for loading and will be truncated: township_range.
  result = obj(*args, **kwargs)


creating aux links for tod 2


c:\Users\toleary\AppData\Local\Programs\ArcGIS\Pro\Resources\ArcPy\arcpy\__init__.py:2969: Warning: The content in the specified field(s) exceeds the defined length for loading and will be truncated: township_range.
  result = obj(*args, **kwargs)


creating aux links for tod 3


...

c:\Users\toleary\AppData\Local\Programs\ArcGIS\Pro\Resources\ArcPy\arcpy\__init__.py:2969: Warning: The content in the specified field(s) exceeds the defined length for loading and will be truncated: township_range.
  result = obj(*args, **kwargs)


creating aux links for tod 3


c:\Users\toleary\AppData\Local\Programs\ArcGIS\Pro\Resources\ArcPy\arcpy\__init__.py:2969: Warning: The content in the specified field(s) exceeds the defined length for loading and will be truncated: township_range.
  result = obj(*args, **kwargs)


creating aux links for tod 4


c:\Users\toleary\AppData\Local\Programs\ArcGIS\Pro\Resources\ArcPy\arcpy\__init__.py:2969: Warning: The content in the specified field(s) exceeds the defined length for loading and will be truncated: township_range.
  result = obj(*args, **kwargs)


In [ ]:
tod_aux = pd.read_csv(rf'C:\Users\toleary\OneDrive - Chicago Metropolitan Agency for Planning\Desktop\test_300_3.csv')

#fill missing zone17 values from the "force" dataset (making centroid-to-stop links)
nulls = tod_aux.loc[tod_aux['zone17'].isnull()].copy()
nulls['zone17'] = nulls[['ANODE','BNODE']].min(axis=1)
tod_aux.update(nulls['zone17'], join='left', overwrite=False)

In [39]:
stop_centroid = tod_aux[24239:].copy()
acc_egr_links = tod_aux[8540:24239].copy()
xfer_links = tod_aux[:8540].copy()

In [48]:
stop_centroid['dist_rank'] = 0
acc_egr_links['dist_rank'] = acc_egr_links['length']
# all_acc_egr = pd.concat([acc_egr_links, stop_centroid])
all_acc_egr = acc_egr_links.copy()

In [58]:
all_acc_egr.drop_duplicates(['ANODE', 'BNODE', 'aux_mode'], inplace=True)
all_acc_egr['count'] = 1
num_connects = all_acc_egr.groupby(['zone17', 'aux_mode']).agg({'count':'sum'}).reset_index()
all_acc_egr = pd.merge(
    all_acc_egr.drop(columns='count'),
    num_connects,
    on=['zone17', 'aux_mode'],
    how='left'
)

all_stops = 'memory/all_stops'
config = get_access_link_config(all_stops)

In [59]:
max_connect_dict = {}
for aux_mode, geog_config in config.items():
    for geog, config in geog_config.items():
        max_connect_dict[aux_mode] = config['max_connections']

all_acc_egr['max_connections'] = all_acc_egr['aux_mode'].map(max_connect_dict)

In [63]:
all_acc_egr.loc[all_acc_egr['max_connections']<all_acc_egr['count']]

,Unnamed: 0,aux_mode,ANODE,BNODE,length,zone17,dist_rank,count,max_connections
3918,17,x,15551,3,149.998292,3.0,149.998292,5,3.0
3919,18,x,15584,3,511.954812,3.0,511.954812,5,3.0
3920,20,x,15549,3,706.354925,3.0,706.354925,5,3.0
3921,24,x,15483,3,719.480226,3.0,719.480226,5,3.0
3922,25,x,15582,3,859.865795,3.0,859.865795,5,3.0
...,...,...,...,...,...,...,...,...,...
7043,218,y,32032,41,2249.018718,41.0,2249.018718,7,3.0
7047,222,y,32032,45,1373.911094,45.0,1373.911094,4,3.0
7048,223,y,31750,45,1582.675445,45.0,1582.675445,4,3.0
7049,224,y,32031,45,2779.369334,45.0,2779.369334,4,3.0


In [ ]:


(
    all_acc_egr
    .sort_values(
        ['count', 'zone17', 'aux_mode', 'dist_rank'], 
        ascending=[False, True, True, False]
    )
    [['count', 'dist_rank', 'zone17', 'aux_mode', 'ANODE', 'BNODE']]
)


,count,dist_rank,zone17,aux_mode,ANODE,BNODE
6856,7,2587.242872,10.0,y,32096,10
6855,7,2565.705774,10.0,y,32092,10
6854,7,2090.803718,10.0,y,32170,10
6853,7,2073.424568,10.0,y,32094,10
6852,7,1937.545365,10.0,y,32070,10
...,...,...,...,...,...,...
486,1,2713.620465,2914.0,u,2914,14860
5080,1,2713.620465,2914.0,x,14860,2914
3752,1,788.574852,3316.0,w,3316,42330
7879,1,788.574852,3316.0,z,42330,3316


In [31]:
#enforce max_connections from access_link_config
stop_centroid['dist_rank'] = 0
acc_egr_links['dist_rank'] = acc_egr_links['length']
all_acc_egr = pd.concat([acc_egr_links, stop_centroid])
all_acc_egr['zone17'] = all_acc_egr[['ANODE', 'BNODE']].min(axis=1)
all_acc_egr.drop_duplicates(['aux_mode', 'ANODE', 'BNODE'], inplace=True)
all_acc_egr.sort_values(['aux_mode', 'ANODE', 'BNODE'], inplace=True)
all_acc_egr['count'] = 1
stop_count = all_acc_egr.groupby(['ANODE', 'BNODE', 'aux_mode']).agg({'count':'sum'}).reset_index()
all_acc_egr = pd.merge(
    all_acc_egr.drop(columns='count'),
    stop_count, 
    on=['ANODE','BNODE','aux_mode'], 
    how='left'
)


In [33]:
stop_count

,ANODE,BNODE,aux_mode,count
0,1,15213,u,1
1,2,15416,u,1
2,3,15483,u,1
3,3,15549,u,1
4,3,15551,u,1
...,...,...,...,...
12663,48003,362,z,1
12664,48004,308,z,1
12665,48004,309,z,1
12666,48004,314,z,1


In [ ]:

# # acc_egr_config = get_access_link_config(all_stops)
# for aux_mode, geog_config in acc_egr_config.items():
#     if aux_mode in all_acc_egr['aux_mode'].unique().tolist():
#         for geog, config in geog_config.items():
#             query = f'(aux_mode == "{aux_mode}")'
#             if config['centroid_where']:
#                 query += f' & ({config['centroid_where']})'
#             idx = all_acc_egr.query(query).index    
#             all_acc_egr.loc[idx, 'max_connections'] = config['max_connections']
            
# all_acc_egr.sort_values(['aux_mode', 'ANODE', 'BNODE'])

In [12]:
all_acc_egr

,Unnamed: 0,aux_mode,ANODE,BNODE,length
0,519,b,15050,15049,459.413422
1,520,b,15070,15069,1290.422119
2,523,b,15098,15077,220.043213
3,532,b,15061,15207,1262.971680
4,533,b,15207,15208,464.260193
...,...,...,...,...,...
29030,15180,z,48003,298,0.053418
29031,15181,z,48004,309,0.119174
29032,15182,z,48005,3389,1.545977
29033,15183,z,49002,683,0.099084


In [ ]:
all_acc_egr.groupby([])

all_acc_egr.sort_values(['zone17', 'aux_mode', 'dist_rank'])[['zone17', 'aux_mode', 'dist_rank', 'max_connections']]

KeyError: 'zone17'

In [ ]:
aux_mode = 'u'
tod_aux.query(f'(ANODE < 3637) & (aux_mode == @aux_mode)')


,aux_mode,ANODE,BNODE,length
9660,u,1,15213,149.998329
9661,u,1,15213,149.998329
9662,u,1,15213,149.998329
9663,u,2,15416,748.047422
9664,u,2,15416,748.047422
...,...,...,...,...
27363,u,1094,24068,0.471231
27364,u,2548,24078,0.051832
27365,u,2649,24087,0.816987
27366,u,2258,24141,0.384962


In [ ]:
#helper function to write transaction file header
def make_transaction_format(in_df, mode_col, ):
    

today_date = dt.date.today().strftime('%d%b%y').upper()
scen = 100
tod = 1
transact_file_header = dedent(
f'''\
    c BASE NETWORK LINK BATCHIN FILE FOR TRANSIT SCENARIO NETWORK {scen} TOD {tod}
    c ACCESS LINKS (modes c, m, u, v, w, x, y, z)
    c {today_date}
    c a, i-node, j-node, length, modes, type, lanes, vdf
'''
)
tod_content = []
for row in 

# write_transaction_header(r"C:\Users\toleary\OneDrive - Chicago Metropolitan Agency for Planning\Desktop\in_it.txt", transact_file_header)